[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTB-MR/qMRData/blob/main/notebooks/qualitative_imaging.ipynb)

In [ ]:
import importlib

if not importlib.util.find_spec("mrpro"):
    %pip install mrpro[notebooks]

## Qualitative Image Reconstruction

Qualitative image reconstruction does not require any sequence specific signal models and so we can use the same image reconstruction code to obtain image data from multiple scanners and field strengths. 
Here we will focus on T1-weighted, T2-weighted and proton density (PD)-weighted images. All the parameters required for image reconstruction are part of the (ISMR)MRD raw data file. 

In [ ]:
import numpy as np
import torch
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import KData, IData, CsmData
from mrpro.data.traj_calculators import KTrajectoryCartesian

### Download data

The following zenodo data sets can be used with this notebook:
- [IBT-CMR (Zurich, Switzerland), Philips Ingenia Ambition X, ramped down to 0.6T](https://zenodo.org/records/18847561)
- [i3M (Valencia, Spain), Physio1](https://zenodo.org/records/18460613)
- [PUC (Santiago de Chile, Chile), Siemens MAGNETOM Free.Max 0.55T](https://doi.org/10.5281/zenodo.19189305)

In [ ]:
# Get the raw data from zenodo
import os
import tempfile
from pathlib import Path
import zipfile
import zenodo_get

tmp = tempfile.TemporaryDirectory()  # RAII, automatically cleaned up
data_folder = Path(tmp.name)
zenodo_get.download(
    record="18847561",
    retry_attempts=5,
    output_dir=data_folder,
    file_glob=("Subject 01.zip",),
)
with zipfile.ZipFile(data_folder / Path("Subject 01.zip"), "r") as zip_ref:
    zip_ref.extractall(data_folder)

# List all files
for root, dirs, files in os.walk(data_folder):
    for name in files:
        print(os.path.join(root, name))

## Select the data files
Each scan session included T1-weighted, T2-weighted and PD-weighted images and we will reconstruct them together. 

In addition to the filenames for the raw and dicom data we can also define here a function which ensures the images
are orientated 

In [ ]:
# ISMRMRD raw data filenames
raw_file_t1w = data_folder / Path(
    "Subject 01/Qualitative protocol/Subject01-Brain/ISMRMRD/Subject01-Brain-T1.h5"
)
raw_file_t2w = data_folder / Path(
    "Subject 01/Qualitative protocol/Subject01-Brain/ISMRMRD/Subject01-Brain-PD.h5"
)
raw_file_pdw = data_folder / Path(
    "Subject 01/Qualitative protocol/Subject01-Brain/ISMRMRD/Subject01-Brain-T2.h5"
)

# DICOM image data filenames
image_folder_t1w = data_folder / Path(
    "Subject 01/Qualitative protocol/Subject01-Brain/Dicom_PhilipsRecon/Subject01-Brain-T1/"
)
image_folder_t2w = data_folder / Path(
    "Subject 01/Qualitative protocol/Subject01-Brain/Dicom_PhilipsRecon/Subject01-Brain-PD/"
)
image_folder_pdw = data_folder / Path(
    "Subject 01/Qualitative protocol/Subject01-Brain/Dicom_PhilipsRecon/Subject01-Brain-T2/"
)

rsa_or_spr = "spr"  # 'rsa' for knee imaging, 'spr' for brain imaging

## Reconstruct images

In [ ]:
kdata = KData.from_file(raw_file_t1w, trajectory=KTrajectoryCartesian())
csm = CsmData.from_kdata_inati(kdata, downsampled_size=64)
recon = DirectReconstruction(kdata, csm=csm)
t1w = recon(kdata)

kdata = KData.from_file(raw_file_t2w, trajectory=KTrajectoryCartesian())
csm = CsmData.from_kdata_inati(kdata, downsampled_size=64)
recon = DirectReconstruction(kdata, csm=csm)
t2w = recon(kdata)

kdata = KData.from_file(raw_file_pdw, trajectory=KTrajectoryCartesian())
csm = CsmData.from_kdata_inati(kdata, downsampled_size=64)
recon = DirectReconstruction(kdata, csm=csm)
pdw = recon(kdata)

## Visualise results

In [ ]:
import matplotlib.pyplot as plt
from einops import rearrange
import nibabel as nib
from typing import Literal

from nibabel.orientations import (
    io_orientation,
    axcodes2ornt,
    ornt_transform,
    apply_orientation,
)


def show_image(
    t1w: IData,
    t2w: IData,
    pdw: IData,
    vmax_percentile: float = 99.9,
    rsa_or_spr: Literal["rsa", "spr"] = "rsa",
) -> None:
    """Show the qualitative images."""
    _, ax = plt.subplots(3, 3, figsize=(8, 8))

    for cax in ax.flatten():
        cax.set_axis_off()

    def orient_images(idata: IData) -> np.array:
        # Ensure the slices are in the correct order
        if idata.header.idx.slice.unique().numel() > 1:
            positions = torch.stack(
                torch.broadcast_tensors(*kdata.header.acq_info.position.zyx), -1
            ).squeeze()
            orientation = torch.stack(
                torch.broadcast_tensors(
                    *kdata.header.acq_info.orientation.as_directions()[0].zyx
                )
            )
            orientation = orientation.squeeze()
            if orientation.ndim == 2:
                orientation = orientation[..., 0]
            sort_idx = torch.argsort((positions @ orientation).squeeze(), stable=True)
            idata = idata[sort_idx]
        orientation = idata.header.orientation.as_matrix().squeeze()
        if orientation.ndim == 3:
            orientation = orientation.mean(0)
        affine_zyx = torch.cat(
            [
                torch.tensor([[1.0, 0.0, 0.0, 0.0]]),
                torch.cat([torch.zeros((3, 1)), orientation], 1),
            ],
            0,
        )

        data = rearrange(
            idata.data, "... other coils z y x-> x y z 1 other (...) coils"
        )
        img_nii = nib.Nifti1Image(
            data.squeeze().abs().numpy(force=True),
            affine_zyx.flip([0, 1]).numpy(),
            dtype=np.float32,
        )
        # Target orientation (RAS)
        ras_ornt = axcodes2ornt(tuple((rsa_or_spr.upper())))
        transform = ornt_transform(io_orientation(img_nii.affine), ras_ornt)
        ras_data = apply_orientation(img_nii.get_fdata(), transform)
        return ras_data

    def plot_multi_slice_image(ax, img, title, vmax_percentile):
        """Plot three slices of M2D/3D image."""
        # Ensure the slices are in the correct order
        idim = img.squeeze().shape[0]
        center = idim // 2
        offset = max(1, idim // 6)
        idx = np.clip([center - offset, center, center + offset], 0, idim - 1)

        for cax, slice_idx in zip(ax, idx):
            vmax = np.percentile(img[slice_idx, :, :], vmax_percentile)
            cax.imshow(
                np.squeeze(img[slice_idx, :, :]),
                cmap="gray",
                vmin=0,
                vmax=vmax,
                origin="lower",
            )
        ax[1].set_title(title)

    plot_multi_slice_image(ax[0], orient_images(t1w), "T1-weighted", vmax_percentile)
    plot_multi_slice_image(ax[1], orient_images(t2w), "T2-weighted", vmax_percentile)
    plot_multi_slice_image(ax[2], orient_images(pdw), "PD-weighted", vmax_percentile)

    plt.tight_layout()
    plt.show()

In [ ]:
show_image(t1w, t2w, pdw, rsa_or_spr=rsa_or_spr)

## Load Dicom data

In [ ]:
t1w_dicom = IData.from_dicom_folder(image_folder_t1w)
t2w_dicom = IData.from_dicom_folder(image_folder_t2w)
pdw_dicom = IData.from_dicom_folder(image_folder_pdw)

## Visualise Dicom images

In [ ]:
show_image(t1w_dicom, t2w_dicom, pdw_dicom, rsa_or_spr=rsa_or_spr)